# `trained_model_xai.py` — Explainability (XAI) Bridge

## Purpose
Wraps the trained RoBERTa-GCN sentiment model with multiple explanation strategies so the
backend's `/explain` endpoint can tell users **why** a particular sentiment was predicted.
The class exposes the same JSON shape as the legacy `ClearViewExplainer` so the frontend
requires zero changes.

## XAI Methods Provided

| Method | Speed | What it shows |
|--------|-------|---------------|
| `explain_ig_aspect()` | Fast | Integrated Gradients — which tokens most drove the prediction for **one aspect** |
| `explain_ig_conflict()` | Medium | Tokens that simultaneously drive **both** positive and negative aspects (conflict drivers) |
| `explain_msr_delta()` | Fast | Probability distribution **before vs. after** temperature scaling |
| `explain_lime_aspect()` | Slow | LIME word-level attribution (perturbs text, fits linear surrogate) |
| `explain_shap_aspect()` | Slow | SHAP Shapley values via `PartitionExplainer` |

## Frontend API Contract
All methods return dicts with `top_tokens: [[token, score], ...]`.
- **Positive score** → token supported the predicted label
- **Negative score** → token pushed against the predicted label

In [ ]:
# ── Imports and path setup ────────────────────────────────────────────────
import sys, os, re

ML_RESEARCH   = os.path.dirname(os.path.abspath(''))  # ml-research/
inference_dir = os.path.join(ML_RESEARCH, 'outputs', 'cosmetic_sentiment_v1', 'evaluation')
src_dir       = os.path.join(ML_RESEARCH, 'src')
for _d in [inference_dir, src_dir]:
    if _d not in sys.path: sys.path.insert(0, _d)

from inference import SentimentPredictor  # noqa: E402

print('[OK] SentimentPredictor imported successfully')

## `TrainedModelXAI` — Class Definition

The class wraps a `SentimentPredictor` instance and adds five explanation methods.
It is designed to be a **drop-in replacement** for the legacy `ClearViewExplainer`.

### Constructor
```python
xai = TrainedModelXAI(checkpoint_path, temperature=0.5)
```
- `checkpoint_path` — path to the `.pt` checkpoint file produced by training
- `temperature` — calibration temperature applied to the logits (lower = more confident)

In [ ]:
class TrainedModelXAI:
    """
    XAI wrapper for the trained RoBERTa-GCN model.

    Exposes the same interface the backend_server.py expects (matching legacy
    ClearViewExplainer output shape) so the frontend requires no changes.

    Attributes
    ----------
    predictor    : SentimentPredictor  — handles tokenisation, forward pass, and attention
    aspect_names : list[str]           — aspects the model can classify (e.g. colour, smell, ...)
    """

    def __init__(self, checkpoint_path: str, temperature: float = 0.5):
        # Load the model once and keep it in memory for all subsequent explain() calls
        self.predictor    = SentimentPredictor(checkpoint_path, temperature=temperature)
        self.aspect_names = self.predictor.aspect_names
        print('[XAI] Trained model XAI bridge loaded.')
        print('      Aspects:', ', '.join(self.aspect_names))

## Method 1 — `explain_ig_aspect()`: Integrated Gradients per Aspect

**How Integrated Gradients (IG) works:**
1. Start at a **baseline** (zero-embedding input — no information).
2. Interpolate from baseline → actual input in `n_steps` steps (Riemann sum).
3. Compute the gradient of the model's output w.r.t. the input embedding at each step.
4. Sum (integrate) the gradients to get each token's attribution.

Result: tokens with **large positive attribution** strongly supported the predicted label;
tokens with **large negative attribution** pushed against it.

> Uses **Captum** (`captum.attr.IntegratedGradients`) internally via `SentimentPredictor.explain_with_integrated_gradients()`.

In [ ]:
    def explain_ig_aspect(self, text: str, aspect: str,
                          enable_msr: bool = True, top_k: int = 10) -> dict:
        """
        Return top-k tokens that most influenced the prediction for `aspect`.
        Uses true Integrated Gradients from Captum.

        Parameters
        ----------
        text       : The review text to explain.
        aspect     : One of the model's aspect names (e.g. 'colour', 'smell').
        enable_msr : Reserved for future MSR-aware explanations (unused now).
        top_k      : How many tokens to return in the result.

        Returns
        -------
        dict with keys:
          top_tokens : [[token, score], ...]  — sorted by |score| descending
          method     : 'attention'            — kept literal so frontend tab routing works
          task       : 'aspect:<name>'
          predicted  : predicted sentiment string
          confidence : float
          probs      : {negative, neutral, positive}  — raw probabilities
        """
        # n_steps=50 is a good trade-off: accurate enough for the completeness axiom
        # (IG sums exactly equal the output difference from baseline to input)
        # without being prohibitively slow.
        ig_res = self.predictor.explain_with_integrated_gradients(
            text=text, aspect=aspect, n_steps=50, top_k=top_k, save_path=None
        )

        # ── Format tokens ────────────────────────────────────────────────────
        top_tokens = []
        for token, attr in zip(ig_res['tokens'], ig_res['attributions']):
            # Skip RoBERTa special tokens and pure whitespace BPE prefixes
            if token not in ('<s>', '</s>', '<pad>', '<mask>') and len(token.strip('Ġ▁ ')) > 0:
                top_tokens.append([self._clean_token(token), round(float(attr), 6)])

        # Sort by absolute magnitude, deduplicate (keep highest-scored occurrence)
        top_tokens.sort(key=lambda x: abs(x[1]), reverse=True)
        seen, deduped = set(), []
        for t, v in top_tokens:
            if t.lower() not in seen:
                seen.add(t.lower())
                deduped.append([t, v])
            if len(deduped) >= top_k:
                break

        # Also fetch raw probabilities to satisfy the frontend's probability display
        pred = self.predictor.predict(text, aspect)

        return {
            'top_tokens': deduped,
            'method'    : 'attention',          # literal kept for frontend compatibility
            'task'      : f'aspect:{aspect}',
            'predicted' : ig_res['target_label'],
            'confidence': round(ig_res['confidence'], 4),
            'probs': {
                'negative': round(pred['probabilities']['negative'], 4),
                'neutral' : round(pred['probabilities']['neutral'],  4),
                'positive': round(pred['probabilities']['positive'], 4),
            }
        }

## Method 2 — `explain_ig_conflict()`: Conflict Driver Detection

**Goal:** Find tokens that appear with high attention in **both** a positively-predicted
aspect AND a negatively-predicted aspect — these are the "conflict drivers" of the review.

**Algorithm:**
```
For each token t:
    pos_sum = sum of attention weights for t across POSITIVE aspects
    neg_sum = sum of attention weights for t across NEGATIVE aspects
    conflict_score(t) = sqrt(pos_sum * neg_sum)   ← geometric mean
```
The geometric mean is high **only when both** `pos_sum` and `neg_sum` are high,
which precisely identifies tokens active in conflicting directions.

Example: in *"The colour is beautiful but the smell is disgusting"*, the word
*"but"* typically has high attention for both the positively-predicted `colour`
aspect and the negatively-predicted `smell` aspect.

In [ ]:
    def explain_ig_conflict(self, text: str,
                            enable_msr: bool = True, top_k: int = 10) -> dict:
        """
        Identify tokens that drive sentiment conflict across aspects.

        Returns
        -------
        dict with keys:
          top_tokens : [[token, conflict_score], ...]  — sorted descending
          method     : 'attention'
          task       : 'conflict'
        """
        # ── Step 1: Gather per-aspect predictions and attention weights ───────
        aspect_attentions = {}   # { aspect: {token: weight} }
        predictions       = {}   # { aspect: 'positive' | 'neutral' | 'negative' }

        for asp in self.aspect_names:
            result          = self.predictor.predict(text, asp, return_attention=True)
            predictions[asp] = result['sentiment']
            if 'attention' in result:
                aspect_attentions[asp] = dict(
                    zip(result['attention']['tokens'], result['attention']['weights'])
                )

        if not aspect_attentions:
            # Model returned no attention data — return empty result
            return {'top_tokens': [], 'method': 'attention', 'task': 'conflict'}

        # ── Step 2: Compute per-token conflict score ───────────────────────
        # Collect all unique non-special tokens across all aspects
        all_tokens = {
            t for d in aspect_attentions.values() for t in d
            if not t.startswith('<') and t not in ('Ġ', '')
        }

        token_conflict: dict[str, float] = {}
        for tok in all_tokens:
            pos_sum = sum(
                aspect_attentions[asp].get(tok, 0.0)
                for asp in aspect_attentions if predictions.get(asp) == 'positive'
            )
            neg_sum = sum(
                aspect_attentions[asp].get(tok, 0.0)
                for asp in aspect_attentions if predictions.get(asp) == 'negative'
            )
            # Geometric mean: high only when both sides are high
            conflict_score = (pos_sum * neg_sum) ** 0.5
            if conflict_score > 0:
                token_conflict[tok] = round(conflict_score, 6)

        # ── Step 3: Sort and format ────────────────────────────────────────
        sorted_tokens = sorted(token_conflict.items(), key=lambda x: x[1], reverse=True)
        top_tokens    = [[self._clean_token(t), s] for t, s in sorted_tokens[:top_k]]

        return {'top_tokens': top_tokens, 'method': 'attention', 'task': 'conflict'}

## Method 3 — `explain_msr_delta()`: Temperature Scaling Visualisation

**What it shows:** The probability distribution at two calibration states:
- **Before** — with temperature `T = 1.0` (raw logits, uncalibrated)
- **After** — with the trained calibration temperature (e.g. `T = 0.5`, making the model more confident)

This helps users understand how temperature scaling sharpens (or flattens) the model's
output probabilities, and is displayed in the frontend's *MSR Delta* panel.

In [ ]:
    def explain_msr_delta(self, text: str, aspect: str, top_k: int = 10) -> dict:
        """
        Show the probability distribution before/after temperature scaling.

        Returns
        -------
        dict with keys:
          prob_before : [neg, neu, pos]  — probabilities at T=1.0 (no scaling)
          prob_after  : [neg, neu, pos]  — probabilities at calibrated temperature
          method      : 'msr_delta'
          task        : 'aspect:<name>'
        """
        # ── 'After' = current calibrated prediction ────────────────────────
        raw_result  = self.predictor.predict(text, aspect)
        probs_after = [
            raw_result['probabilities']['negative'],
            raw_result['probabilities']['neutral'],
            raw_result['probabilities']['positive'],
        ]

        # ── 'Before' = prediction with temperature reset to 1.0 (no calibration) ──
        original_temp            = self.predictor.temperature
        self.predictor.temperature = 1.0  # temporarily disable calibration
        try:
            before_result = self.predictor.predict(text, aspect)
            probs_before  = [
                before_result['probabilities']['negative'],
                before_result['probabilities']['neutral'],
                before_result['probabilities']['positive'],
            ]
        finally:
            # Always restore the original temperature even if an exception occurs
            self.predictor.temperature = original_temp

        return {
            'prob_before': [round(p, 4) for p in probs_before],
            'prob_after' : [round(p, 4) for p in probs_after],
            'method'     : 'msr_delta',
            'task'       : f'aspect:{aspect}',
        }

## Method 4 — `explain_lime_aspect()`: LIME Explanation

**LIME (Local Interpretable Model-agnostic Explanations):**
1. Perturb the input text by randomly dropping words (generating `num_samples` variants).
2. Get the model's probability on each perturbed variant.
3. Fit a **local linear model** (weighted by proximity to original) on those (variant, probability) pairs.
4. The linear model's coefficients tell us which words matter most **locally** near this input.

**Pros:** Model-agnostic, faithful near the input, easy to interpret.  
**Cons:** Slow (calls the model `num_samples` times), may be unstable on short texts.

Requires: `pip install lime`

In [ ]:
    def explain_lime_aspect(self, text: str, aspect: str,
                            num_samples: int = 100, top_k: int = 10) -> dict:
        """
        Explain prediction using LIME.

        Parameters
        ----------
        text        : Review text to explain.
        aspect      : Aspect to explain.
        num_samples : Number of perturbed variants to generate (more = more stable but slower).
        top_k       : Number of top tokens to return.

        Returns
        -------
        dict with keys: top_tokens, method='lime', task, predicted, confidence, probs
        """
        from lime.lime_text import LimeTextExplainer
        import numpy as np

        # Base prediction — determines which class (label index) we explain
        result         = self.predictor.predict(text, aspect)
        pred_sentiment = result['sentiment']
        conf           = result['confidence']
        labels_map     = {'negative': 0, 'neutral': 1, 'positive': 2}
        target_idx     = labels_map.get(pred_sentiment, 2)

        def predictor_fn(texts: list[str]) -> 'np.ndarray':
            """
            Callback that LIME calls with a list of perturbed strings.
            Must return a numpy array of shape (len(texts), n_classes).
            """
            return np.array([
                [
                    res['probabilities']['negative'],
                    res['probabilities']['neutral'],
                    res['probabilities']['positive'],
                ]
                for res in [self.predictor.predict(t, aspect) for t in texts]
            ])

        explainer = LimeTextExplainer(class_names=['negative', 'neutral', 'positive'])
        exp       = explainer.explain_instance(
            text, predictor_fn,
            labels=(target_idx,),
            num_features=top_k,
            num_samples=num_samples
        )

        # Extract word-level attributions for the predicted class
        top_tokens = [
            [word, round(float(score), 6)]
            for word, score in exp.as_list(label=target_idx)
        ]

        return {
            'top_tokens': top_tokens,
            'method'    : 'lime',
            'task'      : f'aspect:{aspect}',
            'predicted' : pred_sentiment,
            'confidence': round(conf, 4),
            'probs': {
                'negative': round(result['probabilities']['negative'], 4),
                'neutral' : round(result['probabilities']['neutral'],  4),
                'positive': round(result['probabilities']['positive'], 4),
            }
        }

## Method 5 — `explain_shap_aspect()`: SHAP Explanation

**SHAP (SHapley Additive exPlanations):**
Computes each word's **Shapley value** — the average marginal contribution of that word
across all possible subsets of words.  This satisfies several fairness axioms (efficiency,
symmetry, dummy, additivity) that simpler attribution methods do not.

Uses `shap.PartitionExplainer` with a whitespace-based text masker, which:
1. Splits text into words.
2. Repeatedly masks/unmasks subsets of words.
3. Approximates Shapley values without evaluating all 2^n subsets.

Requires: `pip install shap`

In [ ]:
    def explain_shap_aspect(self, text: str, aspect: str,
                            max_evals: int = 100, top_k: int = 10) -> dict:
        """
        Explain prediction using SHAP PartitionExplainer.

        Parameters
        ----------
        text      : Review text.
        aspect    : Aspect to explain.
        max_evals : Budget for number of model calls (more = more accurate but slower).
        top_k     : Number of top tokens to return.

        Returns
        -------
        dict with keys: top_tokens, method='shap', task, predicted, confidence, probs
        """
        import shap
        import numpy as np

        result         = self.predictor.predict(text, aspect)
        pred_sentiment = result['sentiment']
        conf           = result['confidence']
        labels_map     = {'negative': 0, 'neutral': 1, 'positive': 2}
        target_idx     = labels_map.get(pred_sentiment, 2)

        def predictor_fn(texts) -> 'np.ndarray':
            """SHAP calls this with an array-like of strings."""
            return np.array([
                [
                    res['probabilities']['negative'],
                    res['probabilities']['neutral'],
                    res['probabilities']['positive'],
                ]
                for res in [self.predictor.predict(str(t), aspect) for t in texts]
            ])

        # Whitespace-based masker: replaces masked words with empty string
        masker   = shap.maskers.Text(r'\W')  # split on non-word characters
        explainer = shap.Explainer(
            predictor_fn, masker,
            output_names=['negative', 'neutral', 'positive']
        )

        try:
            shap_values = explainer([text], max_evals=max_evals)

            # shap_values.data[0]          → list of word tokens
            # shap_values.values[0, :, i]  → Shapley values for class i
            tokens = shap_values.data[0]
            values = shap_values.values[0, :, target_idx]  # values for predicted class

            # Pair tokens with their Shapley values, filter whitespace
            pairs = sorted(
                [(t.strip(), round(float(v), 6)) for t, v in zip(tokens, values) if t.strip()],
                key=lambda x: abs(x[1]), reverse=True
            )

            # Deduplicate preserving the highest-magnitude occurrence
            seen, top_tokens = set(), []
            for t, v in pairs:
                if t.lower() not in seen:
                    seen.add(t.lower())
                    top_tokens.append([t, v])
                if len(top_tokens) >= top_k:
                    break

        except Exception as e:
            print(f'[SHAP] Error: {e}')
            top_tokens = []

        return {
            'top_tokens': top_tokens,
            'method'    : 'shap',
            'task'      : f'aspect:{aspect}',
            'predicted' : pred_sentiment,
            'confidence': round(conf, 4),
            'probs': {
                'negative': round(result['probabilities']['negative'], 4),
                'neutral' : round(result['probabilities']['neutral'],  4),
                'positive': round(result['probabilities']['positive'], 4),
            }
        }

## Helper Methods

### `_top_attention_tokens()`
Extracts and ranks tokens by raw attention weight from the model's last attention layer.
Used internally when no gradient-based attribution is available.

### `_clean_token()`
Strips the RoBERTa BPE prefix character `Ġ` (Unicode U+0120, a capital G with a dot above),
which RoBERTa uses to indicate that a subword token was preceded by a space.
Without cleaning, tokens appear as `Ġbeautiful` instead of `beautiful`.

In [ ]:
    def _top_attention_tokens(self, predict_result: dict, top_k: int) -> list:
        """
        Extract the top-k tokens ranked by attention weight.

        Parameters
        ----------
        predict_result : dict returned by `SentimentPredictor.predict(..., return_attention=True)`
        top_k          : Maximum number of tokens to return.

        Returns
        -------
        list of [token, weight] pairs sorted by weight descending.
        """
        if 'attention' not in predict_result:
            return []

        tokens  = predict_result['attention']['tokens']
        weights = predict_result['attention']['weights']

        # Filter out RoBERTa special tokens and pure BPE-prefix tokens
        pairs = [
            (self._clean_token(t), w)
            for t, w in zip(tokens, weights)
            if t not in ('<s>', '</s>', '<pad>', '<mask>')
            and len(t.strip('Ġ▁ ')) > 0
        ]

        # Sort descending by weight, deduplicate by token string
        pairs.sort(key=lambda x: x[1], reverse=True)
        seen, result = set(), []
        for tok, w in pairs:
            if tok not in seen:
                seen.add(tok)
                result.append([tok, round(float(w), 6)])
            if len(result) >= top_k:
                break

        return result

    @staticmethod
    def _clean_token(token: str) -> str:
        """
        Strip the RoBERTa BPE space-prefix character (Ġ = U+0120) and
        Sentence-Piece prefix (▁ = U+2581) from the beginning of a token.
        """
        return token.lstrip('Ġ▁')

## Quick Test / Verification

Run the cells below to verify that the class can be instantiated and that all five
explain methods return the expected output shape.

> **Requires a trained checkpoint** at `ml-research/outputs/cosmetic_sentiment_v1/best_model.pt`.
> Skip this section if the checkpoint is not present (it is git-ignored due to size).

In [ ]:
import json

# ── Locate checkpoint ────────────────────────────────────────────────────────
CKPT = os.path.join(project_root, 'ml-research', 'outputs',
                    'cosmetic_sentiment_v1', 'best_model.pt')

if not os.path.exists(CKPT):
    print(f'[SKIP] Checkpoint not found at {CKPT}')
    print('       Train the model first or point CKPT to an existing .pt file.')
else:
    # ── Instantiate ──────────────────────────────────────────────────────────
    xai = TrainedModelXAI(CKPT, temperature=0.5)

    # ── Test text with a deliberate sentiment conflict ────────────────────────
    TEXT = 'The colour is absolutely beautiful but the smell is absolutely disgusting.'
    print('Text:', TEXT)
    print()

    # ── Test Method 1: IG Aspect ─────────────────────────────────────────────
    for asp in ['colour', 'smell']:
        ig = xai.explain_ig_aspect(TEXT, asp)
        assert 'top_tokens' in ig and isinstance(ig['top_tokens'], list), 'IG aspect: bad shape'
        print(f'[IG aspect={asp}] predicted={ig["predicted"]}  conf={ig["confidence"]}')
        print(f'  top tokens: {ig["top_tokens"][:3]}')

    print()

    # ── Test Method 2: IG Conflict ───────────────────────────────────────────
    conflict = xai.explain_ig_conflict(TEXT)
    assert 'top_tokens' in conflict, 'IG conflict: missing top_tokens'
    print('[IG conflict] top conflict tokens:', conflict['top_tokens'][:3])

    # ── Test Method 3: MSR Delta ─────────────────────────────────────────────
    delta = xai.explain_msr_delta(TEXT, 'colour')
    assert 'prob_before' in delta and 'prob_after' in delta, 'MSR delta: missing keys'
    print('[MSR delta] prob_before:', delta['prob_before'])
    print('[MSR delta] prob_after :', delta['prob_after'])

    print()
    print('[OK] All three primary XAI methods verified successfully.')
    print('[NOTE] LIME and SHAP tests skipped here (slow, require lime/shap packages).')